In [1]:
import cv2
import torch
from models.src.encoder import YOLODataEncoder
from models.src.yolo import YOLO
from models.utils.data import load_groundtruths

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
input_size=(300, 300)

In [3]:
data_path = "dataset/Dataset/validation/Vehicle registration plate"
data = load_groundtruths(data_path, train=False, shuffle=False, debug=True)
images, boxes, labels, tot_snamples = data
idx = 1
sample = (images[idx], boxes[idx], labels[idx])
sample

Total 386 images and 386 boxes loaded from: dataset/Dataset/validation/Vehicle registration plate
Debug mode enabled: Only using 10 samples for set: validation


('dataset/Dataset/validation/Vehicle registration plate/00723dac8201a83e.jpg',
 [[1.398784, 408.147456, 60.326912, 438.24460799999997],
  [787.3146879999999, 347.72966399999996, 830.90432, 374.35161600000004]],
 [1, 1])

In [4]:
image = cv2.imread(sample[0])
img_size = image.shape
img_size

(768, 1024, 3)

In [5]:
bboxes = sample[1]
bboxes_norm = torch.tensor(bboxes, dtype=torch.float)

bboxes_norm[:, [0, 2]] *= (input_size[1] / img_size[1])
bboxes_norm[:, [1, 3]] *= (input_size[0] / img_size[0])

In [14]:
yolo_model = YOLO(backbone_name= "resnet18",
                 train_backbone= False,
                 returned_layers= [4],
                 num_classes= 2,
                 fpn_channels= 256)

Freezing conv1.weight
Freezing bn1.weight
Freezing bn1.bias
Freezing layer1.0.conv1.weight
Freezing layer1.0.bn1.weight
Freezing layer1.0.bn1.bias
Freezing layer1.0.conv2.weight
Freezing layer1.0.bn2.weight
Freezing layer1.0.bn2.bias
Freezing layer1.1.conv1.weight
Freezing layer1.1.bn1.weight
Freezing layer1.1.bn1.bias
Freezing layer1.1.conv2.weight
Freezing layer1.1.bn2.weight
Freezing layer1.1.bn2.bias
Freezing layer2.0.conv1.weight
Freezing layer2.0.bn1.weight
Freezing layer2.0.bn1.bias
Freezing layer2.0.conv2.weight
Freezing layer2.0.bn2.weight
Freezing layer2.0.bn2.bias
Freezing layer2.0.downsample.0.weight
Freezing layer2.0.downsample.1.weight
Freezing layer2.0.downsample.1.bias
Freezing layer2.1.conv1.weight
Freezing layer2.1.bn1.weight
Freezing layer2.1.bn1.bias
Freezing layer2.1.conv2.weight
Freezing layer2.1.bn2.weight
Freezing layer2.1.bn2.bias
Freezing layer3.0.conv1.weight
Freezing layer3.0.bn1.weight
Freezing layer3.0.bn1.bias
Freezing layer3.0.conv2.weight
Freezing layer

In [15]:
yolo_model.backbone.strides

[32]

In [16]:
image.shape

(300, 300, 3)

In [17]:
image = cv2.resize(image, input_size)
image_tensor = torch.tensor(image, dtype=torch.float).permute(2, 0, 1).unsqueeze(0)
image_tensor.shape

torch.Size([1, 3, 300, 300])

In [18]:
out = yolo_model(image_tensor)

In [19]:
out.shape

torch.Size([1, 100, 7])

In [20]:
encoder = YOLODataEncoder(input_size=input_size, strides=yolo_model.backbone.strides)
encoded = encoder.encode(bboxes_norm, sample[2])
encoded.shape

torch.Size([100, 6])

In [21]:
encoder.decode(encoded, nms_threshold=0.5, score_threshold=0.5)

tensor([[230.6586, 135.8319, 243.4290, 146.2311,   1.0000,   1.0000],
        [  0.4098, 159.4326,  17.6739, 171.1893,   1.0000,   1.0000]])